# Investment AI
## Notebook 03 — Build the Structured Delta Foundation

In Notebook 02, we landed the raw source files into governed Unity Catalog Volumes.

Now we convert the structured source files into governed Delta tables.

The flow is:

```text
Raw CSV files in Unity Catalog Volumes
                ↓
      Explicit schemas + validation
                ↓
          Delta Lake tables
                ↓
      Governed portfolio foundation
```

This notebook creates:

```text
investment_ai.portfolio
├── companies
├── funds
├── holdings
├── prices
├── benchmarks
└── current_positions   ← governed view (holdings with fund and company information)

investment_ai.research
└── document_manifest
```

We intentionally do **not** load private evaluation ground truth such as guidance history.  
That information must remain isolated so our agents cannot "cheat" later.


## Why build Delta tables instead of querying the CSV files directly?

The CSV files are our raw landing layer.

For the application layer, we want:

- explicit data types
- governed Unity Catalog tables
- reliable schemas
- data-quality checks
- lineage
- easier SQL and Genie access
- a stable interface for agents and tools

The raw files remain unchanged in the Volumes, while Delta becomes the trusted structured layer.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    DateType,
)

CATALOG = "investment_ai"
PORTFOLIO_SCHEMA = "portfolio"
RESEARCH_SCHEMA = "research"

SEED_DATA_PATH = f"/Volumes/{CATALOG}/{PORTFOLIO_SCHEMA}/seed_data"
RESEARCH_METADATA_PATH = (
    f"/Volumes/{CATALOG}/{RESEARCH_SCHEMA}/raw_documents/metadata"
)

print("Portfolio seed path :", SEED_DATA_PATH)
print("Research metadata   :", RESEARCH_METADATA_PATH)


## 1. Define explicit source schemas

For a production-style pipeline, we do not want to depend on schema inference for important business data.

Defining the schema up front gives us predictable types and helps catch bad source data early.


In [0]:
companies_schema = StructType([
    StructField("security_id", StringType(), False),
    StructField("company_name", StringType(), False),
    StructField("sector", StringType(), False),
    StructField("country", StringType(), False),
])

funds_schema = StructType([
    StructField("fund_id", StringType(), False),
    StructField("fund_name", StringType(), False),
    StructField("strategy", StringType(), False),
])

holdings_schema = StructType([
    StructField("as_of_date", DateType(), False),
    StructField("fund_id", StringType(), False),
    StructField("security_id", StringType(), False),
    StructField("shares", LongType(), False),
    StructField("market_price_usd", DoubleType(), False),
    StructField("market_value_usd", LongType(), False),
    StructField("portfolio_weight_pct", DoubleType(), False),
])

prices_schema = StructType([
    StructField("price_date", DateType(), False),
    StructField("security_id", StringType(), False),
    StructField("close_price_usd", DoubleType(), False),
])

benchmarks_schema = StructType([
    StructField("benchmark_id", StringType(), False),
    StructField("benchmark_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("base_index", DoubleType(), False),
])

document_manifest_schema = StructType([
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("security_id", StringType(), False),
    StructField("company_name", StringType(), False),
    StructField("sector", StringType(), False),
    StructField("fiscal_quarter", StringType(), False),
    StructField("fiscal_year", IntegerType(), False),
    StructField("document_type", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source", StringType(), False),
    StructField("publish_date", DateType(), False),
    StructField("language", StringType(), False),
    StructField("confidentiality", StringType(), False),
    StructField("relative_path", StringType(), False),
])

print("✅ Explicit schemas defined.")


## 2. Read the landed source files

At this point, the files are still raw sources in Unity Catalog Volumes.

We read them using the explicit schemas defined above.


In [0]:
def read_csv(path: str, schema: StructType):
    return (
        spark.read
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(schema)
        .csv(path)
    )


companies_df = read_csv(
    f"{SEED_DATA_PATH}/companies.csv",
    companies_schema,
)

funds_df = read_csv(
    f"{SEED_DATA_PATH}/funds.csv",
    funds_schema,
)

holdings_df = read_csv(
    f"{SEED_DATA_PATH}/holdings.csv",
    holdings_schema,
)

prices_df = read_csv(
    f"{SEED_DATA_PATH}/prices.csv",
    prices_schema,
)

benchmarks_df = read_csv(
    f"{SEED_DATA_PATH}/benchmarks.csv",
    benchmarks_schema,
)

document_manifest_df = read_csv(
    f"{RESEARCH_METADATA_PATH}/document_manifest.csv",
    document_manifest_schema,
)

print("✅ Raw structured sources loaded.")


## 3. Inspect the structured sources

Before writing anything to Delta, let's quickly inspect the data we loaded.


In [0]:
print("Companies")
display(companies_df)

print("Funds")
display(funds_df)

print("Holdings")
display(holdings_df)

print("Prices")
display(prices_df)

print("Benchmarks")
display(benchmarks_df)

print("Research document manifest")
display(document_manifest_df.limit(10))


## 4. Validate the expected source contract

For this synthetic POC, we know the expected source shape.

These checks help us catch accidental file changes before we publish trusted Delta tables.


In [0]:
expected_counts = {
    "companies": 5,
    "funds": 3,
    "holdings": 15,
    "prices": 30,
    "benchmarks": 5,
    "document_manifest": 45,
}

actual_counts = {
    "companies": companies_df.count(),
    "funds": funds_df.count(),
    "holdings": holdings_df.count(),
    "prices": prices_df.count(),
    "benchmarks": benchmarks_df.count(),
    "document_manifest": document_manifest_df.count(),
}

for name, expected in expected_counts.items():
    actual = actual_counts[name]
    print(
        f"{'✅' if actual == expected else '❌'} "
        f"{name}: {actual} rows"
    )
    assert actual == expected, (
        f"{name}: expected {expected} rows, found {actual}"
    )

print("\n✅ Source row-count contract validated.")


## 5. Validate primary keys and required fields

we want to prove that the trusted layer has clean identifiers.

We validate:

- no null identifiers
- unique company IDs
- unique fund IDs
- unique benchmark IDs
- unique research document IDs
- one holding per fund/security/date
- one price per security/date


In [0]:
def assert_no_nulls(df, columns, name):
    condition = None

    for column in columns:
        current = F.col(column).isNull()
        condition = (
            current
            if condition is None
            else condition | current
        )

    bad_rows = df.filter(condition).count()

    assert bad_rows == 0, (
        f"{name}: found {bad_rows} rows "
        f"with null required fields"
    )


def assert_unique(df, columns, name):
    duplicates = (
        df.groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0, (
        f"{name}: duplicate keys found for {columns}"
    )


assert_no_nulls(
    companies_df,
    ["security_id", "company_name"],
    "companies",
)
assert_unique(
    companies_df,
    ["security_id"],
    "companies",
)

assert_no_nulls(
    funds_df,
    ["fund_id", "fund_name"],
    "funds",
)
assert_unique(
    funds_df,
    ["fund_id"],
    "funds",
)

assert_unique(
    benchmarks_df,
    ["benchmark_id"],
    "benchmarks",
)

assert_unique(
    holdings_df,
    ["as_of_date", "fund_id", "security_id"],
    "holdings",
)

assert_unique(
    prices_df,
    ["price_date", "security_id"],
    "prices",
)

assert_unique(
    document_manifest_df,
    ["document_id"],
    "document_manifest",
)

print("✅ Required fields and key uniqueness validated.")


## 6. Validate relationships between the datasets

A holding should never point to a company or fund that does not exist.

A price should never point to an unknown security.

The research manifest should also reference only known synthetic companies.


In [0]:
unknown_holding_securities = (
    holdings_df
    .join(
        companies_df.select("security_id"),
        "security_id",
        "left_anti",
    )
    .count()
)

unknown_holding_funds = (
    holdings_df
    .join(
        funds_df.select("fund_id"),
        "fund_id",
        "left_anti",
    )
    .count()
)

unknown_price_securities = (
    prices_df
    .join(
        companies_df.select("security_id"),
        "security_id",
        "left_anti",
    )
    .count()
)

unknown_document_securities = (
    document_manifest_df
    .join(
        companies_df.select("security_id"),
        "security_id",
        "left_anti",
    )
    .count()
)

assert unknown_holding_securities == 0
assert unknown_holding_funds == 0
assert unknown_price_securities == 0
assert unknown_document_securities == 0

print("✅ Holdings → Companies relationship is valid.")
print("✅ Holdings → Funds relationship is valid.")
print("✅ Prices → Companies relationship is valid.")
print("✅ Research manifest → Companies relationship is valid.")


## 7. Validate portfolio calculations

The source includes both shares/price and market value.

We use that redundancy as a data-quality check:

```text
shares × market price ≈ market value
```

A small tolerance is allowed because some synthetic share counts are rounded to whole shares.


In [0]:
holding_math_errors = (
    holdings_df
    .withColumn(
        "calculated_market_value",
        F.col("shares") * F.col("market_price_usd"),
    )
    .withColumn(
        "difference_usd",
        F.abs(
            F.col("market_value_usd")
            - F.col("calculated_market_value")
        ),
    )
    .filter(F.col("difference_usd") > 100)
)

assert holding_math_errors.count() == 0, (
    "Holdings market-value validation failed."
)

print("✅ shares × price agrees with market value.")


## 8. Publish the trusted Delta tables

These writes use `overwrite` intentionally so the notebook is repeatable during the POC.

For a real production ingestion pipeline, we would choose the write strategy based on the source pattern — for example append, MERGE, CDC, or streaming ingestion.

For this small synthetic dataset, we also do **not** add unnecessary partitioning or optimization.


In [0]:
delta_tables = {
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.companies": companies_df,
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.funds": funds_df,
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.holdings": holdings_df,
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.prices": prices_df,
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.benchmarks": benchmarks_df,
    f"{CATALOG}.{RESEARCH_SCHEMA}.document_manifest": document_manifest_df,
}

for table_name, dataframe in delta_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"✅ {table_name}")


## 9. Add table descriptions

Good metadata matters later when humans, Genie, agents, and other tools discover these assets.

We add simple table comments now rather than leaving the tables undocumented.


In [0]:
table_comments = {
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.companies":
        "Synthetic company master for the Investment AI educational POC.",

    f"{CATALOG}.{PORTFOLIO_SCHEMA}.funds":
        "Synthetic Investment AI fund master.",

    f"{CATALOG}.{PORTFOLIO_SCHEMA}.holdings":
        "Synthetic fund-level security holdings and portfolio exposure snapshots.",

    f"{CATALOG}.{PORTFOLIO_SCHEMA}.prices":
        "Synthetic historical security closing prices.",

    f"{CATALOG}.{PORTFOLIO_SCHEMA}.benchmarks":
        "Synthetic benchmark master used by the Investment AI POC.",

    f"{CATALOG}.{RESEARCH_SCHEMA}.document_manifest":
        "Governed metadata describing the synthetic investment research corpus.",
}

for table_name, comment in table_comments.items():
    escaped = comment.replace("'", "''")
    spark.sql(
        f"COMMENT ON TABLE {table_name} IS '{escaped}'"
    )

print("✅ Table descriptions added.")


## 10. Create a business-friendly `current_positions` view

The raw holdings table is normalized and intentionally simple.

For analytics, Genie, and later agent tools, it is useful to have one governed view that joins:

- holdings
- fund names
- company names
- sectors

The view still keeps the fund-level rows.  
We are **not** summing portfolio weights into a fake firm-wide portfolio weight.


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW
{CATALOG}.{PORTFOLIO_SCHEMA}.current_positions
COMMENT 'Current synthetic fund-level positions enriched with company and fund metadata.'
AS
WITH latest_snapshot AS (
    SELECT MAX(as_of_date) AS as_of_date
    FROM {CATALOG}.{PORTFOLIO_SCHEMA}.holdings
)
SELECT
    h.as_of_date,
    h.fund_id,
    f.fund_name,
    f.strategy,
    h.security_id,
    c.company_name,
    c.sector,
    c.country,
    h.shares,
    h.market_price_usd,
    h.market_value_usd,
    h.portfolio_weight_pct
FROM {CATALOG}.{PORTFOLIO_SCHEMA}.holdings h
INNER JOIN latest_snapshot s
    ON h.as_of_date = s.as_of_date
INNER JOIN {CATALOG}.{PORTFOLIO_SCHEMA}.funds f
    ON h.fund_id = f.fund_id
INNER JOIN {CATALOG}.{PORTFOLIO_SCHEMA}.companies c
    ON h.security_id = c.security_id
""")

print(
    f"✅ Created "
    f"{CATALOG}.{PORTFOLIO_SCHEMA}.current_positions"
)


## 11. Validate the trusted business facts

These checks are useful because they give us known facts that later agent evaluations can rely on.

For the flagship synthetic scenario:

```text
IA-SYN-01 → $220M across all funds
IA-SYN-02 →  $85M across all funds
Combined  → $305M
```

These are portfolio facts only.

Whether a company **lowered guidance** must come from the research side later — not from this portfolio layer.


In [0]:
company_exposure = (
    spark.table(
        f"{CATALOG}.{PORTFOLIO_SCHEMA}.current_positions"
    )
    .groupBy("security_id", "company_name")
    .agg(
        F.sum("shares").alias("total_shares"),
        F.sum("market_value_usd").alias(
            "total_market_value_usd"
        ),
    )
    .orderBy("security_id")
)

display(company_exposure)

exposure = {
    row["security_id"]: row["total_market_value_usd"]
    for row in company_exposure.collect()
}

shares = {
    row["security_id"]: row["total_shares"]
    for row in company_exposure.collect()
}

assert exposure["IA-SYN-01"] == 220_000_000
assert exposure["IA-SYN-02"] == 85_000_000
assert (
    exposure["IA-SYN-01"]
    + exposure["IA-SYN-02"]
    == 305_000_000
)

assert shares["IA-SYN-01"] == 2_000_000
assert shares["IA-SYN-02"] == 1_062_500

print("✅ IA-SYN-01 exposure = $220M")
print("✅ IA-SYN-02 exposure = $85M")
print("✅ Combined exposure  = $305M")


## 12. Inspect the final governed objects


In [0]:
print("Portfolio objects")
display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{PORTFOLIO_SCHEMA}"
    )
)

print("Research objects")
display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{RESEARCH_SCHEMA}"
    )
)

print("Current positions")
display(
    spark.table(
        f"{CATALOG}.{PORTFOLIO_SCHEMA}.current_positions"
    )
)


## Structured foundation complete

We now have a trusted structured layer:

```text
Raw source files
      ↓
Explicit schemas
      ↓
Data-quality validation
      ↓
Unity Catalog Delta tables
      ↓
Business-friendly current_positions view
```

The portfolio side is now ready for later:

- Unity Catalog business semantics
- Metric Views
- Genie
- governed UC Function tools
- Portfolio Agent

We have **not** parsed the research documents yet.

That is the next part of the project.

**Next:** `04_ingest_and_parse_documents.ipynb`


In [0]:
required_objects = {
    "companies",
    "funds",
    "holdings",
    "prices",
    "benchmarks",
    "current_positions",
}

portfolio_objects = {
    row["tableName"]
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.{PORTFOLIO_SCHEMA}"
    ).collect()
}

missing = required_objects - portfolio_objects

assert not missing, f"Missing portfolio objects: {missing}"

assert spark.catalog.tableExists(
    f"{CATALOG}.{RESEARCH_SCHEMA}.document_manifest"
)

print("✅ Structured Delta foundation is ready.")
print("✅ Portfolio relationships and calculations validated.")
print("✅ current_positions view created.")
print("✅ Research document manifest published.")
print("\nNext: 04_ingest_and_parse_documents.ipynb")
